In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('../')

import os
import numpy as np
import pandas as pd
import json
import re
from tqdm import tqdm

from src.utilities_screen_peak_combined import *        # use this instead to not substract mean

import warnings
warnings.filterwarnings("ignore")

protein_ctrl = 'NPC'
compound_ctrl = 'NCC'

In [35]:
working_dir = '/Users/behnam/Desktop/My_Codes/screen_analysis/data_test/bug_fix/CGG-327/'

In [36]:
# read the layout file and convert it into the well_positions_dict
layout = pd.read_excel(os.path.join(working_dir,"TSA_Layout.xlsx"), index_col=0)
well_positions_dict = layout2dict(layout, working_dir)

In [37]:
# Read the raw data
remove_header(os.path.join(working_dir,'TSA_RawData.csv'))

try:    # type I data with header being removed
    rawdata = pd.read_csv(os.path.join(working_dir,'TSA_RawData_removed_header.csv'), index_col=0, encoding='ISO-8859-1')
except: # type II and IV data with ; as separator and extra rows to be removed
    try:
        rawdata = pd.read_csv(os.path.join(working_dir,'TSA_RawData.csv'), encoding='ISO-8859-1', sep=';', index_col = 0)
        rawdata = rawdata.drop(['SampleID'], axis=1)
        pattern = '^[A-Z][0-9]{1,2}$'
        to_keep = [bool(re.findall(pattern, str(i))) for i in rawdata.index]
        rawdata = rawdata.loc[to_keep]
        if len(rawdata.iloc[0,0].split(',')) == 1:       # II
            rawdata = rawdata.apply(pd.to_numeric, errors='coerce')
        else:                                            # IV
            rawdata = rawdata.apply(lambda x: x.str.replace(',', '.').astype(float))
            rawdata.columns = rawdata.columns.str.replace(',', '.')
    except: # type III data with ; as separator and headers
        remove_header(os.path.join(working_dir,'TSA_RawData.csv'), sep=';')
        rawdata = pd.read_csv(os.path.join(working_dir,'TSA_RawData_removed_header.csv'), index_col=0, encoding='ISO-8859-1', sep=';')
        rawdata = rawdata.apply(pd.to_numeric, errors='coerce')

In [30]:
rawdata

,23,23.7,24.4,25.1,25.8,26.5,27.2,27.9,28.6,29.3,...,53.8,54.5,55.2,55.9,56.6,57.3,58,58.7,59.4,60.1
Cy5,,,,,,,,,,,,,,,,,,,,,
A1,4950.153420,4993.119486,5019.288830,4997.487698,4948.314033,4912.191766,4845.590504,4743.359430,4660.145120,4547.489964,...,2649.987542,2592.005580,2532.370565,2487.256329,2465.436563,2398.428028,2335.963391,2278.228988,2201.299473,2139.955505
A2,68815.948022,68839.620484,68888.086079,68787.396258,68490.983376,68240.917200,68273.677461,68326.295652,68148.918550,68008.622008,...,67065.104133,66358.273620,65987.457386,65733.914741,65294.077559,64602.555754,63910.078318,63322.587044,62714.875819,61972.817445
A3,77160.955685,76835.325559,76727.967626,76765.386248,76406.897370,76230.723457,76117.381526,76038.740396,75843.054602,76055.081928,...,68174.587454,67449.501487,67093.355100,66419.391324,65723.530587,64786.515785,64128.933472,63219.788982,62987.975597,62736.314930
A4,76377.115799,76655.094878,76734.505303,76652.235975,76655.137469,76555.709398,76210.979990,75939.299015,75661.256050,75186.944330,...,66673.926866,66212.027684,65692.090639,65093.432607,64435.429710,63786.549215,63039.726002,62450.118501,62005.761378,61627.244207
A5,78127.392699,77772.372304,77509.065733,77599.363415,77832.185667,77919.278396,78037.989422,78045.581221,77866.614952,77254.263999,...,70878.095578,70335.995380,69732.029477,69177.857468,68599.893209,67861.891602,67018.360801,66286.300602,65427.836424,64839.919243
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P20,72912.087342,72912.802764,72918.858784,72805.241180,72543.531310,72381.793630,72349.374097,72235.831680,72080.398368,72030.893018,...,65122.060028,64619.496877,64087.701691,63422.008750,62786.037703,62190.964162,61576.991189,61226.637396,61035.865904,60898.334080
P21,78855.349552,78649.959764,78336.832956,78155.528268,77922.148181,77712.618517,77576.512980,77387.286252,77140.306319,77152.108497,...,67978.979891,67587.664795,67079.448902,66575.302254,65779.933456,65062.304099,64399.415884,63914.988898,63522.491306,63191.084792
P22,71792.236127,71653.154983,71591.797330,71408.881805,71173.790630,71064.404038,70943.042109,70806.615316,70603.412804,70590.170670,...,64469.757318,63890.726082,63404.544717,62849.999581,62318.594005,61628.606350,60896.308571,60163.386508,59581.422954,59077.604398


In [31]:
# Get all proteins, ligands and doses
protein_set = list(well_positions_dict.keys())
compound_set = list(well_positions_dict[protein_set[0]].keys())
try:
    dosage_set = list(well_positions_dict[protein_set[0]][compound_set[0]].keys())
except:
    dosage_set = list(well_positions_dict[protein_set[0]][compound_set[1]].keys())
protein_set, compound_set, dosage_set


(['GCDH'],
 ['A02',
  'A03',
  'A04',
  'A05',
  'A06',
  'A07',
  'A08',
  'A09',
  'A10',
  'A11',
  'A12',
  'A13',
  'A14',
  'A15',
  'A16',
  'A17',
  'A18',
  'A19',
  'A20',
  'A21',
  'A22',
  'needle1-buf',
  'B01',
  'B02',
  'B03',
  'B04',
  'B05',
  'B06',
  'B07',
  'B08',
  'B09',
  'B10',
  'B11',
  'B12',
  'B13',
  'B14',
  'B15',
  'B16',
  'B17',
  'B18',
  'B19',
  'B20',
  'B21',
  'B22',
  'needle1-prot',
  'C02',
  'C03',
  'C04',
  'C05',
  'C06',
  'C07',
  'C08',
  'C09',
  'C10',
  'C11',
  'C12',
  'C13',
  'C14',
  'C15',
  'C16',
  'C17',
  'C18',
  'C19',
  'C20',
  'C21',
  'C22',
  'needle2-buf',
  'D01',
  'D02',
  'D03',
  'D04',
  'D05',
  'D06',
  'D07',
  'D08',
  'D09',
  'D10',
  'D11',
  'D12',
  'D13',
  'D14',
  'D15',
  'D16',
  'D17',
  'D18',
  'D19',
  'D20',
  'D21',
  'D22',
  'needle2-prot',
  'E02',
  'E03',
  'E04',
  'E05',
  'E06',
  'E07',
  'E08',
  'E09',
  'E10',
  'E11',
  'E12',
  'E13',
  'E14',
  'E15',
  'E16',
  'E17',
 

In [32]:
# Get max_replicate
max_replicate = 0
for protein in protein_set:
    for compound in compound_set:
        for dosage in dosage_set:
            # try:
            well_positions_target = well_positions_dict[protein][compound][dosage] if compound != compound_ctrl else well_positions_dict[protein][compound]   
            # except:
            # well_positions_target = []
            if len(well_positions_target) > max_replicate:
                max_replicate = len(well_positions_target)
        
colnames = (['protein', 'compound', 'dosage', 'locations'] +
            [f'tm1_{i}' for i in range(1, max_replicate+1)] + ['tm_mean', 'tm_std'] +
            [f'tm2_{i}' for i in range(1, max_replicate+1)] + ['tm_mean', 'tm_std'] +
            [f'r_squared_{i}' for i in range(1, max_replicate+1)] +
            # [f'peak_count_{i}' for i in range(1, max_replicate+1)] + ['peak_count_mean', 'peak_count_std'] +
            [f'fl25_{i}' for i in range(1, max_replicate+1)]
       )

In [33]:
fit_curve = False
df = pd.DataFrame(columns=colnames)
curve = pd.DataFrame()

# Loop over protein
for protein in tqdm(protein_set):
    if protein == protein_ctrl:
        continue
    # Loop over compound
    for compound in compound_set:
        # check for 0 dosage
        if compound == compound_ctrl:
            row, crv = process2(well_positions_dict, rawdata,
                                protein, compound,
                                save_dir = working_dir,
                                fit_curve = fit_curve,
                                max_replicate = max_replicate)
            df.loc[len(df)] = row
            curve = pd.concat([curve, crv], axis=1)
        else:
            # Loop over dosage
            for dosage in dosage_set:
                row, crv = process2(well_positions_dict, rawdata, 
                                    protein, compound, dosage, 
                                    save_dir = working_dir,
                                    fit_curve = fit_curve,
                                    max_replicate = max_replicate)
                df.loc[len(df)] = row
                curve = pd.concat([curve, crv], axis=1)

curve.index = crv['Temperature']
curve = curve.drop('Temperature', axis = 1)

100%|█████████████████████████████████████████████| 1/1 [00:44<00:00, 44.82s/it]


In [26]:
df['tm2_1'].isna().sum()

np.int64(0)

In [38]:
df.to_csv(os.path.join(working_dir, "result_table.csv"))

In [34]:
df

,protein,compound,dosage,locations,tm1_1,tm_mean,tm_std,tm2_1,tm_mean,tm_std,r_squared_1,fl25_1
0,GCDH,A02,dosage,A2,40.4539,40.4539,0.0,48.3889,48.3889,0.0,0.0036,68815.9480215978
1,GCDH,A03,dosage,A3,45.3902,45.3902,0.0,57.0,57.0,0.0,0.0075,77160.9556845266
2,GCDH,A04,dosage,A4,45.1451,45.1451,0.0,46.3468,46.3468,0.0,0.0084,76377.1157988578
3,GCDH,A05,dosage,A5,44.2958,44.2958,0.0,54.9214,54.9214,0.0,0.0063,78127.3926985281
4,GCDH,A06,dosage,A6,44.3188,44.3188,0.0,55.899,55.899,0.0,0.006,52217.9286831909
...,...,...,...,...,...,...,...,...,...,...,...,...
353,GCDH,P18,dosage,P18,43.8398,43.8398,0.0,54.165,54.165,0.0,0.0019,80074.4415873887
354,GCDH,P19,dosage,P19,43.8331,43.8331,0.0,55.7864,55.7864,0.0,0.003,79085.2094919135
355,GCDH,P20,dosage,P20,44.6678,44.6678,0.0,55.2102,55.2102,0.0,0.0025,72912.0873420198
356,GCDH,P21,dosage,P21,44.6882,44.6882,0.0,57.0,57.0,0.0,0.0088,78855.3495522021
